In [ ]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.1 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="PASTE_YOUR_GROQ_KEY_HERE",
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq model connected!")

Groq model connected!


In [ ]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]      # running conversation
    name: str                # remembered user name
    preferences: str         # remembered preferences

print("ChatState created successfully!")

ChatState created successfully!


In [ ]:
def chatbot(state: ChatState) -> dict:
    user_msg = state["messages"][-1]

    # Get existing memory
    name = state.get("name", "")
    prefs = state.get("preferences", "")

    # Remember the user's name
    if "my name is" in user_msg.lower():
        name = user_msg.lower().split("my name is")[-1].strip().title()

    # Remember what the user likes
    if "i like" in user_msg.lower():
        prefs = user_msg.lower().split("i like")[-1].strip()

    # Give the LLM the remembered information
    context = f"The user's name is {name}. They like {prefs}."

    reply = llm.invoke(
        context + "\nReply naturally to: " + user_msg
    ).content

    return {
        "messages": state["messages"] + [reply],
        "name": name,
        "preferences": prefs
    }

print("Chatbot node created successfully!")

Chatbot node created successfully!


In [ ]:
from langgraph.graph import StateGraph, START, END

# Create the graph builder
builder = StateGraph(ChatState)

# Add our chatbot node
builder.add_node("chatbot", chatbot)

# Define the flow
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Compile the graph
graph = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="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",
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq model connected!")

Groq model connected!


In [ ]:
test = llm.invoke("Say hello in one short sentence.")
print(test.content)

Hello.


In [ ]:
state = {
    "messages": [],
    "name": "",
    "preferences": ""
}

while True:
    user = input("You: ")

    if user.lower() == "quit":
        break

    state["messages"].append(user)

    state = graph.invoke(state)

    print("Bot:", state["messages"][-1])


You: I am Manasa.V. i like to sing
Bot: Nice to meet you, Manasa.V. Singing is such a wonderful hobby - what kind of music do you enjoy singing the most? Are you more into classical, pop, or something else?
You: carntaic
Bot: Carntaic, that's a unique name. Do you have a favorite song to sing, or perhaps a favorite artist that inspires your voice?
You: Thank you
Bot: You're welcome. I'm glad I could help. Do you have a favorite song to sing?
You: quit
